# Retail Sales & Customer Performance Analysis
**Author:** Babli  
**Tools:** Python (Pandas, NumPy, Matplotlib), SQL, Microsoft Excel

## 1. Project Overview
This notebook demonstrates end-to-end data loading, data cleaning, exploratory data analysis (EDA), SQL aggregation queries, and data visualization on 2,500 retail sales transactions.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

# Load dataset
df = pd.read_csv('data/retail_sales_data.csv')
print('Dataset shape:', df.shape)
df.head()

## 2. Data Cleaning & Inspection
Check for missing null values, duplicate records, and verify column data types.

In [2]:
# Check missing values
print('Missing values:\n', df.isnull().sum())
print('\nDuplicate Orders:', df['Order_ID'].duplicated().sum())
df.info()

## 3. Feature Engineering
Parse `Order_Date` to datetime format and engineer `Month`, `Month_Name`, and `Day_Name` columns.

In [3]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.strftime('%b')
df['Day_Name'] = df['Order_Date'].dt.day_name()
df[['Order_ID', 'Order_Date', 'Month_Name', 'Day_Name', 'Total_Revenue']].head()

## 4. Key Business KPIs

In [4]:
total_revenue = df['Total_Revenue'].sum()
total_orders = len(df)
total_quantity = df['Quantity'].sum()
aov = total_revenue / total_orders

print(f'Total Revenue:       INR {total_revenue:,.2f}')
print(f'Total Orders:        {total_orders:,}')
print(f'Total Units Sold:    {total_quantity:,}')
print(f'Average Order Value: INR {aov:,.2f}')

## 5. Visualizations & Category Breakdown

In [5]:
# Category Performance
cat_summary = df.groupby('Product_Category').agg(
    Revenue=('Total_Revenue', 'sum'),
    Orders=('Order_ID', 'count'),
    Units=('Quantity', 'sum')
).reset_index().sort_values('Revenue', ascending=False)

plt.figure(figsize=(9, 4.5))
plt.barh(cat_summary['Product_Category'], cat_summary['Revenue']/1000, color='#2b6cb0')
plt.title('Revenue by Product Category (in Thousands INR)', fontweight='bold')
plt.xlabel('Revenue (K INR)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. SQL Queries with In-Memory Database
Load data into SQLite to run business intelligence queries.

In [6]:
conn = sqlite3.connect(':memory:')
df.to_sql('retail_sales', conn, index=False, if_exists='replace')

query = '''
SELECT Product_Category, 
       COUNT(Order_ID) AS Total_Orders,
       SUM(Quantity) AS Total_Units,
       ROUND(SUM(Total_Revenue), 2) AS Category_Revenue
FROM retail_sales
GROUP BY Product_Category
ORDER BY Category_Revenue DESC;
'''
pd.read_sql_query(query, conn)